In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

BASE_DIR = r"G:\jupyter\2025APMCM"
TARIFF_DIR = os.path.join(BASE_DIR, "Tariff Data")
TRADE_DIR = os.path.join(BASE_DIR, "Trade Data")
OUTPUT_DIR = os.path.join(BASE_DIR, "Cleaned_Data")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SECTORS = {'Soybean': ['1201', '120190'], 'Auto': ['8703'], 'Chips': ['8541', '8542']}

def clean_rate(s):
    s = str(s).lower().replace('%', '').replace('free', '0').replace('nan', '0').replace('nm', '0')
    return pd.to_numeric(s, errors='coerce')

def get_trade_weights():
    files = ["US_Import_Cars_Chips_2015_2024.csv", "Soybean_Competition_Exports_to_China_2015_2024.csv"]
    dfs = []
    for f in files:
        p = os.path.join(TRADE_DIR, f)
        if os.path.exists(p):
            d = pd.read_csv(p, usecols=lambda c: c in ['refYear', 'cmdCode', 'primaryValue', 'fobvalue'])
            val_col = 'primaryValue' if 'primaryValue' in d.columns else 'fobvalue'
            d = d.rename(columns={'refYear': 'Year', 'cmdCode': 'hts', val_col: 'Value'})
            d['hts_4'] = d['hts'].astype(str).str.replace('.', '').str.strip().str.slice(0, 4)
            dfs.append(d[['Year', 'hts_4', 'Value']])
    return pd.concat(dfs).groupby(['Year', 'hts_4'])['Value'].sum().reset_index() if dfs else pd.DataFrame()

weights = get_trade_weights()
results = []
sector_data = []

for year in range(2015, 2026):
    path = os.path.join(TARIFF_DIR, f"tariff_data_{year}", f"tariff_database_{year}")
    f_path = path + ".xlsx" if os.path.exists(path + ".xlsx") else path + ".txt"
    if not os.path.exists(f_path): continue
    
    if f_path.endswith('.xlsx'):
        df = pd.read_excel(f_path, dtype={'hts8': str})
    else:
        sep = ',' if year == 2025 else '|'
        df = pd.read_csv(f_path, sep=sep, encoding='latin1', dtype={'hts8': str}, on_bad_lines='skip')
    
    df.columns = [c.lower().strip() for c in df.columns]
    if 'mfn_ad_val_rate' not in df.columns: continue
    
    df['hts8'] = df['hts8'].astype(str).str.replace('.', '').str.strip()
    df['Rate'] = df['mfn_ad_val_rate'].apply(clean_rate).fillna(0.0)
    df = df[df['Rate'] < 10.0]
    
    res = {'Year': year, 'General_Tau': df['Rate'].mean()}
    w_year = min(year, 2024)
    t_w = weights[weights['Year'] == w_year]
    df['hts_4'] = df['hts8'].str.slice(0, 4)
    merged = pd.merge(df.groupby('hts_4')['Rate'].mean().reset_index(), t_w, on='hts_4')
    
    for sec, prefs in SECTORS.items():
        p_list = [p[:4] for p in prefs]
        sec_d = merged[merged['hts_4'].isin(p_list)]
        if not sec_d.empty and sec_d['Value'].sum() > 0:
            res[f'{sec}_Tau'] = (sec_d['Rate'] * sec_d['Value']).sum() / sec_d['Value'].sum()
        else:
            res[f'{sec}_Tau'] = df[df['hts_4'].isin(p_list)]['Rate'].mean()
        
        sec_slice = df[df['hts_4'].isin(p_list)][['hts8', 'Rate']].copy()
        sec_slice['Sector'] = sec
        sec_slice['Year'] = year
        if len(sec_slice) > 10:
            clf = IsolationForest(random_state=42)
            mask = clf.fit_predict(sec_slice[['Rate']]) == 1
            sec_slice = sec_slice[mask]
        sector_data.append(sec_slice)
        
    res['Simulated_Shock'] = max(res['General_Tau'], 0.10) if year >= 2025 else res['General_Tau']
    results.append(res)

pd.concat(sector_data).to_csv(os.path.join(OUTPUT_DIR, "Sector_Base_Tariffs.csv"), index=False)
final_df = pd.DataFrame(results)
final_df.to_csv(os.path.join(OUTPUT_DIR, "Annual_Tariff_Index.csv"), index=False)

plt.figure(figsize=(12, 6))
sns.lineplot(data=final_df, x='Year', y='General_Tau', marker='o', label='Historical')
sns.lineplot(data=final_df[final_df['Year']>=2024], x='Year', y='Simulated_Shock', color='red', label='Shock')
plt.savefig(os.path.join(OUTPUT_DIR, "Figure_2_Tariff_Shock_Index.png"))
plt.close()

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.integrate import odeint
from scipy.optimize import minimize
import geopandas as gpd
from shapely.geometry import Polygon
from shapely.affinity import scale, translate

BASE_DIR = r"G:\jupyter\2025APMCM"
TRADE_FILE = os.path.join(BASE_DIR, "Trade Data", "Soybean_Competition_Exports_to_China_2015_2024.csv")

df = pd.read_csv(TRADE_FILE)
df['Year'] = df['refPeriodId'].astype(str).str.slice(0, 4).astype(int)
country_map = {'USA': 'USA', 'United States': 'USA', 'Brazil': 'BRA', 'BRA': 'BRA', 'Argentina': 'ARG'}
df['Country'] = df['reporterISO'].map(country_map)
df['Value_Billion'] = df['fobvalue'] / 1e9
history_data = df.groupby(['Year', 'Country'])['Value_Billion'].sum().unstack(fill_value=0).reindex(range(2015, 2025)).fillna(0)

def lv_model(x, t, params):
    U, B, A = x
    r_u, r_b, r_a, a_ub, a_ua, a_bu, a_ba, a_au, a_ab = params
    K = 160.0
    U, B, A = max(U, 0), max(B, 0), max(A, 0)
    return [r_u * U * (1 - (U + a_ub*B + a_ua*A)/K),
            r_b * B * (1 - (B + a_bu*U + a_ba*A)/K),
            r_a * A * (1 - (A + a_au*U + a_ab*B)/K)]

def objective(params, t, data):
    return np.mean((odeint(lv_model, data[0], t, args=(params,)) - data)**2)

res = minimize(objective, [0.3]*3 + [0.1]*6, args=(np.arange(len(history_data)), history_data.values),
               bounds=[(0.05, 1.5)]*3 + [(0, 2.0)]*6, method='L-BFGS-B')
params = res.x

def calc_shock(p, tariff=0.2):
    new_p = list(p)
    new_p[0] *= (1/(1+tariff)) * 0.5 
    new_p[3] += np.log(1 + tariff * 2.5) * 0.5
    new_p[1] *= (1 + tariff * 0.5)
    return new_p

shock_params = calc_shock(params)
t_fut = np.arange(7)
fut_base = odeint(lv_model, history_data.iloc[-1], t_fut, args=(params,))
fut_shock = odeint(lv_model, history_data.iloc[-1], t_fut, args=(tuple(shock_params),))
df_shock = pd.DataFrame(fut_shock, index=range(2024, 2031), columns=['USA', 'BRA', 'ARG'])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
history_data.plot(ax=axes[0], marker='o')
df_shock.plot(ax=axes[1], marker='s')
plt.savefig(os.path.join(BASE_DIR, "Cleaned_Data", "Figure_3_Soybean_Competition.png"))
plt.close()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(history_data['USA'], history_data['BRA'], history_data['ARG'], color='gray', label='History')
ax.plot(df_shock['USA'], df_shock['BRA'], df_shock['ARG'], color='red', label='Forecast')
plt.savefig(os.path.join(BASE_DIR, "Cleaned_Data", "Figure_4_Phase_Portrait.png"))
plt.close()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from statsmodels.tsa.api import VARMAX
from statsmodels.tsa.stattools import adfuller

BASE_DIR = r"G:\jupyter\2025APMCM"
MACRO_DIR = os.path.join(BASE_DIR, "US_Macroeconomic")
IMG_DIR = os.path.join(BASE_DIR, "Cleaned_Data")

files = {'US_CPI_Total': 'US_CPI_2015_2025.csv', 'Chip_PPI': 'US_PCU33443344_2015_2025.csv',
         'Auto_PPI': 'US_PCU336336_2015_2025.csv', 'Auto_CPI': 'US_CUSR0000SETA01_2015_2025.csv',
         'JPY_Rate': 'US_DEXJPUS_2015_2025.csv'}
dfs = []
for k, v in files.items():
    p = os.path.join(MACRO_DIR, v)
    if os.path.exists(p):
        d = pd.read_csv(p)
        d.iloc[:, 0] = pd.to_datetime(d.iloc[:, 0])
        d = d.set_index(d.columns[0]).resample('ME').mean().interpolate()
        d.columns = [k]
        dfs.append(d)

df_norm = pd.concat(dfs, axis=1).dropna()
df_norm = df_norm / df_norm.iloc[0] * 100
df_stat = pd.DataFrame()
for c in df_norm.columns:
    df_stat[c] = df_norm[c].diff().dropna() if adfuller(df_norm[c].diff().dropna())[1] < 0.05 else df_norm[c].diff().diff().dropna()

model = VARMAX(df_stat[['Auto_PPI', 'Auto_CPI']].dropna(), exog=df_stat[['Chip_PPI']].loc[df_stat[['Auto_PPI', 'Auto_CPI']].dropna().index], order=(3,0)).fit(disp=False)
irf = model.impulse_responses(steps=24, impulse=0)
plt.figure(); irf.plot(); plt.savefig(os.path.join(IMG_DIR, "Figure_7_VARX_IRF.png")); plt.close()

model_h = VARMAX(df_stat[['Auto_PPI']], exog=df_stat[['Chip_PPI', 'JPY_Rate']], order=(3,0)).fit(disp=False)
irf_c = model_h.impulse_responses(12, impulse=0)
irf_j = model_h.impulse_responses(12, impulse=1)
hedge_ratio = -irf_c['Auto_PPI'].max() / irf_j['Auto_PPI'].min()

costs = {"Export": 30000 * (1 + 0.1 + 0.05 * 0.25), "USA": 30000 * 1.15, "Mexico": 30000 * 1.08}
pd.Series(costs).plot(kind='bar', color=['gray', 'blue', 'green'])
plt.savefig(os.path.join(IMG_DIR, "Figure_9_Scenario.png")); plt.close()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import statsmodels.api as sm
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

BASE_DIR = r"G:\jupyter\2025APMCM"
df_soy = pd.read_csv(os.path.join(BASE_DIR, "Trade Data", "Soybean_Competition_Exports_to_China_2015_2024.csv"))
df_ac = pd.read_csv(os.path.join(BASE_DIR, "Trade Data", "US_Import_Cars_Chips_2015_2024.csv"))
df_gdp = pd.read_csv(os.path.join(BASE_DIR, "US_Macroeconomic", "US_GDP_2015_2025.csv"))
df_tau = pd.read_csv(os.path.join(BASE_DIR, "Cleaned_Data", "Annual_Tariff_Index.csv"))

v1 = df_soy[df_soy['reporterISO']=='USA'].groupby(df_soy['refPeriodId'].astype(str).str[:4].astype(int))['fobvalue'].sum()
v2 = df_ac.groupby(df_ac['refPeriodId'].astype(str).str[:4].astype(int))['primaryValue'].sum()
df_vol = pd.DataFrame({'Vol': (v1+v2)/1e9}).dropna()
df_gdp['Year'] = pd.to_datetime(df_gdp.iloc[:,0]).dt.year
gdp = df_gdp.groupby('Year').mean().iloc[:,0]
data = pd.concat([df_vol, gdp, df_tau.set_index('Year')['General_Tau']], axis=1).dropna()

mod = sm.tsa.SARIMAX(data['Vol'], exog=data.iloc[:,1], order=(1,0,0)).fit(disp=False)
fut_gdp = [data.iloc[-1,1] * 1.02**i for i in range(1, 5)]
fcst = mod.get_forecast(4, exog=np.array(fut_gdp).reshape(-1,1)).predicted_mean
plt.plot(data.index, data['Vol']); plt.plot(range(2025, 2029), fcst); plt.savefig(os.path.join(BASE_DIR, "Cleaned_Data", "Figure_10_Forecast.png")); plt.close()

def laffer(vol, base_t, rates, elast):
    revs = []
    for t in rates:
        v_new = vol * (1 + elast * (t - base_t)/(1+base_t))
        revs.append(max(0, v_new) * t)
    return revs
plt.plot(np.linspace(0, 0.6, 50), laffer(fcst.iloc[0], 0.025, np.linspace(0, 0.6, 50), -1.5))
plt.savefig(os.path.join(BASE_DIR, "Cleaned_Data", "Figure_11S_Laffer.png")); plt.close()

class LSTM(nn.Module):
    def __init__(self): super().__init__(); self.l = nn.LSTM(1, 50, batch_first=True); self.f = nn.Linear(50, 1)
    def forward(self, x): return self.f(self.l(x)[0][:,-1,:])

ts = data['Vol'].values.reshape(-1, 1)
sc = MinMaxScaler((-1, 1)); ts_s = sc.fit_transform(ts)
X, y = [], []
for i in range(len(ts_s)-3): X.append(ts_s[i:i+3]); y.append(ts_s[i+3])
X, y = torch.FloatTensor(X), torch.FloatTensor(y)
m = LSTM(); opt = torch.optim.Adam(m.parameters(), 0.01)
for _ in range(100): opt.zero_grad(); (m(X)-y).pow(2).mean().backward(); opt.step()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas_datareader.data as web
import os
from sklearn.preprocessing import MinMaxScaler

BASE_DIR = r"G:\jupyter\2025APMCM"
IMG_DIR = os.path.join(BASE_DIR, "Cleaned_Data")

payoffs = np.array([[-34, -15], [-15, -25], [0, -5], [0, 0]])
sns.heatmap(payoffs.reshape(2,2), annot=True, fmt='d', cmap='coolwarm')
plt.savefig(os.path.join(IMG_DIR, "Figure_12_Payoff.png")); plt.close()

df = web.DataReader(['INDPRO', 'PAYEMS', 'IMPGS', 'BOPGSTB'], 'fred', '2015-01-01', '2024-12-31').ffill()
df_n = pd.DataFrame(MinMaxScaler((0.01, 1)).fit_transform(df), index=df.index, columns=df.columns)
df_n['IMPGS'] = 1 - df_n['IMPGS']
P = df_n / df_n.sum()
w = (1 + (P * np.log(P)).sum() / np.log(len(df_n))) / ((1 + (P * np.log(P)).sum() / np.log(len(df_n))).sum())
idx = (df_n * w).sum(axis=1) * 100
idx.plot(); plt.savefig(os.path.join(IMG_DIR, "Figure_13_Index.png")); plt.close()

decomp = df_n * w
plt.stackplot(decomp.index, decomp.T)
plt.plot(idx.index, idx/100*w.sum(), 'k')
plt.savefig(os.path.join(IMG_DIR, "Figure_14_Decomp.png")); plt.close()

try:
    d_c = pd.concat([pd.read_csv(os.path.join(BASE_DIR, "US_Macroeconomic", f)).set_index('observation_date') for f in ["Chip_High_IC.csv", "Chip_Low_Discrete.csv", "Chip_Production.csv"]], axis=1).dropna()
    (d_c/d_c.iloc[0]*100).plot()
    plt.savefig(os.path.join(IMG_DIR, "Figure_16_Chips.png")); plt.close()
except: pass

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.integrate import odeint
from scipy.optimize import minimize
from statsmodels.tsa.api import VAR
import plotly.graph_objects as go

BASE_DIR = r"G:\jupyter\2025APMCM"
CLEAN_DIR = os.path.join(BASE_DIR, "Cleaned_Data")
df_soy = pd.read_csv(os.path.join(BASE_DIR, "Trade Data", "Soybean_Competition_Exports_to_China_2015_2024.csv"))
df_soy['Year'] = df_soy['refPeriodId'].astype(str).str[:4].astype(int)
iso_map = {'USA': 'USA', 'Brazil': 'BRA', 'Argentina': 'ARG'}
data = df_soy.assign(C=df_soy['reporterISO'].map(iso_map)).groupby(['Year', 'C'])['fobvalue'].sum().unstack().fillna(0)/1e9

def lv(x, t, p): 
    return [p[0]*x[0]*(1-(x[0]+p[3]*x[1]+p[4]*x[2])/160), p[1]*x[1]*(1-(x[1]+p[5]*x[0]+p[6]*x[2])/160), p[2]*x[2]*(1-(x[2]+p[7]*x[0]+p[8]*x[1])/160)]
def obj(p, t, d): return np.mean((odeint(lv, d[0], t, args=(p,)) - d)**2)

p1 = minimize(obj, [0.5]*9, args=(np.arange(3), data.loc[2015:2017].values), bounds=[(0,2)]*9).x
p2 = minimize(obj, [0.5]*9, args=(np.arange(7), data.loc[2018:2024].values), bounds=[(0,2)]*9).x
pd.DataFrame({'Pre': p1, 'Post': p2}).plot(kind='bar')
plt.savefig(os.path.join(CLEAN_DIR, "Figure_19_Params.png")); plt.close()

d17, d18 = data.loc[2017], data.loc[2018]
fig = go.Figure(data=[go.Sankey(node=dict(label=["US", "BRA", "ARG", "CHN"]), link=dict(source=[0,1,2], target=[3,3,3], value=d17.values))])
fig.write_html(os.path.join(CLEAN_DIR, "Figure_17_Sankey.html"))

df_m = pd.concat([pd.read_csv(os.path.join(BASE_DIR, f)).iloc[:,[0,1]] for f in [r"Cleaned_Data\Annual_Tariff_Index.csv", r"US_Macroeconomic\US_GDP_2015_2025.csv", r"US_Macroeconomic\US_CPI_2015_2025.csv"]], axis=1)
df_m = df_m.apply(pd.to_numeric, errors='coerce').pct_change().dropna()
irf = VAR(df_m).fit(1).irf(5)
plt.figure(); irf.plot(); plt.savefig(os.path.join(CLEAN_DIR, "Figure_18_Macro.png")); plt.close()